# ChurnGuard AI

**Customer Churn Prediction and Retention Recommendation System**

**IBM Project Submission Notebook**

This end-to-end project generates a realistic subscription customer dataset, explores churn behavior, engineers business features, trains and evaluates machine-learning models, explains high-risk customers, and estimates the financial impact of retention campaigns.

**Business objective:** identify customers likely to churn early enough for Customer Success teams to take personalized action.

## 1. Import libraries and configure reproducibility

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, roc_curve, precision_recall_curve
)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')
print('Environment ready')

## 2. Generate a realistic synthetic customer dataset

The dataset contains 20,000 customers and includes missing values, outliers, behavioral signals, and a moderate churn rate. Churn is intentionally more likely for monthly contracts, low engagement, poor satisfaction, payment failures, support burden, price increases, and long login recency.

In [ ]:
n = 20000
signup_dates = pd.to_datetime('2025-01-01') - pd.to_timedelta(rng.integers(30, 2400, n), unit='D')
tenure = np.maximum(1, ((pd.Timestamp('2026-09-01') - signup_dates).days // 30).astype(int))
plan = rng.choice(['Basic', 'Standard', 'Premium'], n, p=[0.35, 0.45, 0.20])
contract = rng.choice(['Monthly', 'Yearly'], n, p=[0.62, 0.38])
payment = rng.choice(['Card', 'UPI', 'Bank', 'Wallet'], n, p=[0.45, 0.25, 0.20, 0.10])
plan_charge = pd.Series(plan).map({'Basic': 35, 'Standard': 65, 'Premium': 110}).to_numpy()
monthly = np.maximum(15, plan_charge + rng.normal(0, 9, n))
auto_pay = (rng.random(n) < np.where(payment == 'Card', .78, .52)).astype(int)
tickets = rng.poisson(1.7, n)
resolution = np.maximum(1, rng.gamma(2.2, 8, n))
satisfaction = np.clip(4.1 - tickets * .16 + rng.normal(0, .65, n), 1, 5)
login_count = rng.poisson(np.clip(15 - tickets * .6, 2, None), n)
session_minutes = np.maximum(5, login_count * rng.normal(18, 6, n))
feature_usage = np.clip(login_count * 3.4 + rng.normal(32, 18, n), 0, 100)
days_last_login = np.maximum(0, rng.gamma(2.2, 8, n) + tickets * 1.2 - login_count * .25)
discount = (rng.random(n) < .28).astype(int)
price_increase = (rng.random(n) < .12).astype(int)
payment_failures = rng.poisson(np.where(auto_pay == 1, .25, .75), n)
referrals = rng.poisson(np.where(satisfaction >= 4, 1.2, .35), n)
total_charges = monthly * tenure * rng.normal(1, .08, n)
billing_cycle = pd.to_datetime(signup_dates).dayofyear % 30
logit = (
    -2.25 + 1.15 * (contract == 'Monthly') + .42 * (monthly > 90)
    - .035 * feature_usage + .035 * days_last_login + .20 * tickets
    - .72 * (satisfaction - 3) + .34 * payment_failures
    + .72 * price_increase - .025 * login_count + .018 * resolution
    + .20 * (billing_cycle > 24) + rng.normal(0, .65, n)
)
churn_probability = 1 / (1 + np.exp(-logit))
churn = (rng.random(n) < churn_probability).astype(int)

df = pd.DataFrame({
    'customer_id': [f'C{i:06d}' for i in range(n)],
    'signup_date': signup_dates.strftime('%Y-%m-%d'),
    'tenure_months': tenure,
    'plan_type': plan,
    'monthly_charges': monthly.round(2),
    'total_charges': total_charges.round(2),
    'contract_type': contract,
    'payment_method': payment,
    'auto_pay': auto_pay,
    'num_support_tickets_30d': tickets,
    'avg_resolution_time_hours': resolution.round(2),
    'satisfaction_score': satisfaction.round(2),
    'login_count_30d': login_count,
    'session_minutes_30d': session_minutes.round(2),
    'feature_usage_score': feature_usage.round(2),
    'days_since_last_login': days_last_login.round(1),
    'discount_applied': discount,
    'price_increase_last_3m': price_increase,
    'num_payment_failures_6m': payment_failures,
    'referral_count': referrals,
    'churn': churn,
})

# Add realistic missing values and a few billing outliers.
for column in ['satisfaction_score', 'feature_usage_score', 'avg_resolution_time_hours', 'monthly_charges']:
    missing_rows = rng.choice(n, size=int(.02 * n), replace=False)
    df.loc[missing_rows, column] = np.nan
outlier_rows = rng.choice(n, size=30, replace=False)
df.loc[outlier_rows, 'monthly_charges'] *= rng.uniform(2.0, 4.0, len(outlier_rows))
df.loc[outlier_rows, 'total_charges'] *= rng.uniform(1.5, 3.0, len(outlier_rows))

print(f'Rows: {len(df):,}')
print(f'Churn rate: {df.churn.mean():.2%}')
display(df.head())

## 3. Exploratory data analysis

In [ ]:
display(df.describe(include='all').T)
display(df.isna().sum().sort_values(ascending=False).head(10).to_frame('missing_values'))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(data=df, x='churn', ax=axes[0])
axes[0].set_title('Churn distribution')
sns.barplot(data=df, x='contract_type', y='churn', ax=axes[1], errorbar=None)
axes[1].set_title('Churn rate by contract')
sns.barplot(data=df, x='plan_type', y='churn', ax=axes[2], errorbar=None)
axes[2].set_title('Churn rate by plan')
plt.tight_layout()
plt.show()

display(df.groupby('payment_method')['churn'].agg(['count', 'mean']).sort_values('mean', ascending=False))

## 4. Feature engineering

In [ ]:
features = df.copy()
features['signup_month'] = pd.to_datetime(features['signup_date']).dt.month
features = features.drop(columns=['signup_date', 'customer_id'])
features['tenure_bucket'] = pd.cut(features['tenure_months'], [-1, 6, 12, 24, 48, np.inf], labels=['0-6', '7-12', '13-24', '25-48', '49+'])
features['charges_per_month'] = features['total_charges'] / features['tenure_months'].clip(lower=1)
features['usage_trend'] = features['login_count_30d'] * .4 + features['session_minutes_30d'] * .01 + features['feature_usage_score'] * .6
features['engagement_score'] = features['login_count_30d'].clip(upper=30) / 30 * 40 + features['session_minutes_30d'].clip(upper=1200) / 1200 * 30 + features['feature_usage_score'] * .3
features['support_load'] = features['num_support_tickets_30d'] * features['avg_resolution_time_hours']
features['payment_reliability'] = 1 / (1 + features['num_payment_failures_6m'])
features['recency_score'] = 1 / (1 + features['days_since_last_login'])
features['loyalty_score'] = features['tenure_months'].clip(upper=60) / 60 * .7 + features['referral_count'].clip(upper=10) / 10 * .3
display(features[['tenure_bucket', 'charges_per_month', 'usage_trend', 'engagement_score', 'support_load', 'payment_reliability', 'recency_score', 'loyalty_score', 'churn']].head())

## 5. Preprocessing and model training

Numeric variables use median imputation and scaling. Categorical variables use most-frequent imputation and one-hot encoding. Class weights are used to make the model more attentive to churners.

In [ ]:
X = features.drop(columns='churn')
y = features['churn']
categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_columns = [c for c in X.columns if c not in categorical_columns]

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_columns),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical_columns),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, stratify=y, random_state=RANDOM_STATE)
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=180, min_samples_leaf=4, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE),
}
results = {}
fitted = {}
for name, estimator in models.items():
    model = Pipeline([('preprocessor', preprocessor), ('classifier', estimator)])
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_test)[:, 1]
    prediction = (probability >= .5).astype(int)
    results[name] = {
        'accuracy': accuracy_score(y_test, prediction),
        'precision': precision_score(y_test, prediction, zero_division=0),
        'recall': recall_score(y_test, prediction, zero_division=0),
        'f1': f1_score(y_test, prediction, zero_division=0),
        'roc_auc': roc_auc_score(y_test, probability),
        'pr_auc': average_precision_score(y_test, probability),
    }
    fitted[name] = model

results_table = pd.DataFrame(results).T.sort_values('roc_auc', ascending=False)
display(results_table.style.format('{:.3f}'))

## 6. Cross-validation and final model selection

ROC-AUC is the primary ranking metric and recall is monitored because missing a likely churner can be costly.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, model in fitted.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_rows.append({'model': name, 'mean_cv_roc_auc': scores.mean(), 'std_cv_roc_auc': scores.std()})
cv_table = pd.DataFrame(cv_rows).sort_values('mean_cv_roc_auc', ascending=False)
display(cv_table.style.format({'mean_cv_roc_auc': '{:.3f}', 'std_cv_roc_auc': '{:.3f}'}))
best_name = results_table.index[0]
best_model = fitted[best_name]
best_probability = best_model.predict_proba(X_test)[:, 1]
best_prediction = (best_probability >= .5).astype(int)
print(f'Selected model: {best_name}')

## 7. Final evaluation: ROC, PR curve, confusion matrix, and decile lift

In [ ]:
print(classification_report(y_test, best_prediction, zero_division=0))
fpr, tpr, _ = roc_curve(y_test, best_probability)
precision, recall, _ = precision_recall_curve(y_test, best_probability)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(fpr, tpr, label=f'AUC={roc_auc_score(y_test, best_probability):.3f}')
axes[0].plot([0, 1], [0, 1], '--', color='gray')
axes[0].set_title('ROC curve'); axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate'); axes[0].legend()
axes[1].plot(recall, precision, label=f'PR-AUC={average_precision_score(y_test, best_probability):.3f}')
axes[1].set_title('Precision-recall curve'); axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].legend()
sns.heatmap(confusion_matrix(y_test, best_prediction), annot=True, fmt='d', cmap='Blues', ax=axes[2])
axes[2].set_title('Confusion matrix'); axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')
plt.tight_layout(); plt.show()

evaluation = X_test.copy()
evaluation['actual_churn'] = y_test.to_numpy()
evaluation['churn_probability'] = best_probability
evaluation['monthly_charges_original'] = df.loc[X_test.index, 'monthly_charges'].to_numpy()
evaluation['decile'] = pd.qcut(evaluation['churn_probability'], 10, labels=False, duplicates='drop') + 1
deciles = evaluation.groupby('decile').agg(customers=('actual_churn', 'size'), churners=('actual_churn', 'sum'), avg_probability=('churn_probability', 'mean'), monthly_revenue=('monthly_charges_original', 'sum')).reset_index()
deciles['churn_rate'] = deciles['churners'] / deciles['customers']
deciles['lift'] = deciles['churn_rate'] / y_test.mean()
display(deciles.sort_values('decile', ascending=False).style.format({'avg_probability': '{:.3f}', 'churn_rate': '{:.2%}', 'lift': '{:.2f}'}))

## 8. Feature importance and churn reason codes

For a transparent baseline, the coefficients of the logistic-regression model are used as global directional reason codes. Positive coefficients increase churn likelihood; negative coefficients reduce it.

In [ ]:
logistic_model = fitted['Logistic Regression']
encoded_names = logistic_model.named_steps['preprocessor'].get_feature_names_out()
coefficients = logistic_model.named_steps['classifier'].coef_[0]
importance = pd.DataFrame({'feature': encoded_names, 'coefficient': coefficients})
importance['absolute_effect'] = importance['coefficient'].abs()
top_importance = importance.sort_values('absolute_effect', ascending=False).head(20)
display(top_importance)
plt.figure(figsize=(10, 7))
sns.barplot(data=top_importance.sort_values('coefficient'), x='coefficient', y='feature', palette='vlag')
plt.title('Top global churn drivers')
plt.tight_layout(); plt.show()

def reason_codes(customer_row, top_n=5):
    transformed = logistic_model.named_steps['preprocessor'].transform(pd.DataFrame([customer_row]))
    contribution = np.asarray(transformed.toarray() if hasattr(transformed, 'toarray') else transformed)[0] * coefficients
    reasons = pd.DataFrame({'feature': encoded_names, 'contribution': contribution})
    return reasons.reindex(reasons.contribution.abs().sort_values(ascending=False).index).head(top_n)

sample_customer = X_test.iloc[0].copy()
display(reason_codes(sample_customer))


## 9. Personalized retention recommendation engine

In [ ]:
def recommend_offer(customer, probability):
    if probability >= .70:
        offer = '20% loyalty discount and personal success-manager call'
        risk = 'High'
    elif probability >= .40:
        offer = '10% discount for 6 months with plan review'
        risk = 'Medium'
    else:
        offer = 'Product education, usage tips, and light-touch nurture'
        risk = 'Low'
    reasons = []
    if customer.get('satisfaction_score', 5) <= 2.5: reasons.append('low satisfaction')
    if customer.get('num_support_tickets_30d', 0) >= 4: reasons.append('high support burden')
    if customer.get('days_since_last_login', 0) >= 21: reasons.append('long login recency')
    if customer.get('num_payment_failures_6m', 0) >= 2: reasons.append('payment failures')
    if customer.get('contract_type') == 'Monthly': reasons.append('monthly contract')
    return {'risk_level': risk, 'churn_probability': round(float(probability), 4), 'reasons': reasons, 'recommended_offer': offer}

raw_customer = df.loc[X_test.index[0]].drop('churn').to_dict()
display(pd.Series(recommend_offer(raw_customer, best_probability[0])))


## 10. Revenue-at-risk and campaign ROI

This scenario targets the highest-risk 10% of customers. The assumptions are explicit and can be changed for a real business.

In [ ]:
scored = df.drop(columns='churn').copy()
scored['churn_probability'] = best_model.predict_proba(X)[:, 1]
campaign = scored.sort_values('churn_probability', ascending=False).head(int(.10 * len(scored)))
retention_success_rate = .25
months_saved = 6
cost_per_contact = 12.0
gross_revenue_saved = (campaign['monthly_charges'] * campaign['churn_probability'] * retention_success_rate * months_saved).sum()
campaign_cost = len(campaign) * cost_per_contact
net_value = gross_revenue_saved - campaign_cost
roi = net_value / campaign_cost
business_summary = pd.Series({
    'customers_targeted': len(campaign),
    'estimated_gross_revenue_saved': gross_revenue_saved,
    'campaign_cost': campaign_cost,
    'estimated_net_value': net_value,
    'ROI_multiple': roi,
})
display(business_summary.to_frame('value').style.format('{:,.2f}'))


## 11. Conclusion

The model provides an early-warning ranking rather than an automatic decision. Customer Success teams should combine the probability score with the reason codes, customer value, service history, and offer cost before contacting a customer.

**Recommended operational workflow:**

1. Refresh customer features weekly or daily.
2. Rank customers by churn probability and monthly revenue at risk.
3. Review the top reason codes before outreach.
4. Match the intervention to the driver: service recovery, payment support, product education, discount, or contract migration.
5. Track offer acceptance, retained revenue, and campaign ROI.

This notebook is self-contained and does not require the separate application files to reproduce the core analysis.